# Quantum correlation corrections CP tags
## Calculate quantum correlation correction factors for CP tags

### Include library for handling uncertainties
#### [Here is the ```uncertainties-cpp``` library on GitHub](https://github.com/Gattocrucco/uncertainties-cpp)

In [1]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/uncertainties-cpp");

In [2]:
#include<uncertainties/impl.hpp>
#include<uncertainties/ureal.hpp>
#include<uncertainties/io.hpp>
#include<uncertainties/math.hpp>
#include<uncertainties/stat.hpp>

### Load utility functions

In [3]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Number of bins

In [4]:
const int NumberBins = 4;

### Get fitted values of $c_i$ and $K_i$

In [5]:
const std::string cisiFilename("BiasCorrectedResults.txt");
auto cisiFitted = ParseParameters(cisiFilename);
std::map<int, uncertainties::udouble> ci, si, Ki, Kbari;
for(int Bin = 1; Bin <= NumberBins; Bin++) {
    const std::string BinName = std::to_string(Bin);
    ci[Bin] = uncertainties::udouble(cisiFitted["c" + BinName], cisiFitted["c" + BinName + "_err"]);
    si[Bin] = uncertainties::udouble(cisiFitted["s" + BinName], cisiFitted["s" + BinName + "_err"]);
    Ki[Bin] = uncertainties::udouble(cisiFitted["K" + BinName], cisiFitted["K" + BinName + "_err"]);
    Kbari[Bin] = uncertainties::udouble(cisiFitted["Kbar" + BinName], cisiFitted["Kbar" + BinName + "_err"]);
}

### Get non-normalised predicted bin yield

In [6]:
uncertainties::udouble GetPredictedBinYield(int Bin, const uncertainties::udouble &TagFPlus) {
    return Ki[Bin] + Kbari[Bin] - 2.0*uncertainties::sqrt(Ki[Bin]*Kbari[Bin])*(2.0*TagFPlus - 1.0)*ci[Bin];
}

### List of tags and their backgrounds

In [7]:
const std::map<std::string, std::vector<std::string>> Tags{
    {"pipipi0", std::vector<std::string>{"KSpi0"}},
    {"KSpi0", std::vector<std::string>{"pipipi0"}},
    {"pipipi0PartReco", std::vector<std::string>{"KSpi0PartReco"}},
    {"KSpi0PartReco", std::vector<std::string>{"pipipi0PartReco"}},
    {"KSeta", std::vector<std::string>{"pipieta"}},
    {"KSetaPrimerhogamma", std::vector<std::string>{"KSpipipi0"}},
    {"KSpi0pi0", std::vector<std::string>{"pipipi0pi0", "KSKS", "KSpi0gamma"}},
    {"KLpi0", std::vector<std::string>{"KLpi0pi0", "pi0pi0", "pi0eta", "KSpi0"}}
};

### Start calculating the quantum correlation factors

In [8]:
std::string QuantumCorrelationFactors;
for(const auto &Tag : Tags) {
    auto FPlus = GetFPlus(Tag.first);
    if(FPlus.first == -1.0) {
        std::cout << "Cannot find F+ for " << Tag.first << "\n";
        continue;
    }
    uncertainties::udouble TagFPlus(FPlus.first, FPlus.second);
    for(std::size_t i = 0; i < Tag.second.size(); i++) {
        FPlus = GetFPlus(Tag.second[i]);
        if(FPlus.first == -1.0) {
            std::cout << "Cannot find F+ for " << Tag.second[i] << "\n";
            continue;
        }
        uncertainties::udouble BkgFPlus(FPlus.first, FPlus.second);
        std::vector<uncertainties::udouble> QCFactors;
        for(int Bin = 1; Bin <= NumberBins; Bin++) {
            std::string Label = Tag.first + "_PeakingBackground" + std::to_string(i + 1);
            Label += "_DoubleTag_CP_KKpipi_vs_" + Tag.first + "_SignalBin";
            Label += std::to_string(Bin) + "_QuantumCorrelationFactor";
            auto SigBinYield = GetPredictedBinYield(Bin, TagFPlus);
            auto BkgBinYield = GetPredictedBinYield(Bin, BkgFPlus);
            QCFactors.push_back(BkgBinYield/SigBinYield);
            QuantumCorrelationFactors += Label + " ";
            QuantumCorrelationFactors += std::to_string(uncertainties::nom(QCFactors.back())) + "\n";
            QuantumCorrelationFactors += Label + "_err ";
            QuantumCorrelationFactors += std::to_string(uncertainties::sdev(QCFactors.back())) + "\n";
        }
        QuantumCorrelationFactors += "\n";
        std::string Filename = "PeakingBackground_DT_" + Tag.second[i] + "_to_" + Tag.first;
        Filename += "_QuantumCorrelationFactors.root";
        std::vector<double> FlatCovMatrix =
            uncertainties::cov_matrix<std::vector<double>>(QCFactors);
        SaveCovMatrix(FlatCovMatrix, Filename);
    }
}
//std::cout << QuantumCorrelationFactors;

### Save parameters to a file

In [9]:
std::ofstream File("QuantumCorrelationFactors_CP.txt");
File << QuantumCorrelationFactors;
File.close();

### Do the same for the $K_SK\pi$ background in $K\pi\pi\pi$

In [10]:
uncertainties::udouble deltaD_KSKpi(0.1, 15.7);
deltaD_KSKpi *= TMath::Pi()/180.0;
uncertainties::udouble R_KSKpi(0.70, 0.08);
uncertainties::udouble rD2_KSKpi(0.592, TMath::Sqrt(0.044*0.044 + 0.018*0.018));
uncertainties::udouble rD_KSKpi = uncertainties::sqrt(rD2_KSKpi);

In [22]:
std::cout << rD_KSKpi << "\n";

0.769 ± 0.031


In [13]:
uncertainties::udouble deltaD_Kpipipi(161, 28);
deltaD_Kpipipi *= TMath::Pi()/180.0;
uncertainties::udouble R_Kpipipi(0.44, 0.10);
uncertainties::udouble rD_Kpipipi(0.0550, 0.0007);
uncertainties::udouble rD2_Kpipipi = rD_Kpipipi*rD_Kpipipi;

### Function for calculating the quantum correlation correction

In [11]:
uncertainties::udouble GetKSKpiQCFactor(int Bin) {
    auto sqrtKK = uncertainties::sqrt(Ki[TMath::Abs(Bin)]*Kbari[TMath::Abs(Bin)]);
    auto cosDeltaD = uncertainties::cos(deltaD_KSKpi);
    auto sinDeltaD = uncertainties::sin(deltaD_KSKpi);
    if(Bin > 0) {
        auto Phases = ci[Bin]*cosDeltaD + si[Bin]*sinDeltaD;
        return 1 - 2.0*rD_KSKpi*R_KSKpi*sqrtKK*Phases/(Kbari[Bin] + rD2_KSKpi*Ki[Bin]);
    } else {
        auto Phases = ci[-Bin]*cosDeltaD - si[-Bin]*sinDeltaD;
        return 1 - 2.0*rD_KSKpi*R_KSKpi*sqrtKK*Phases/(Ki[-Bin] + rD2_KSKpi*Kbari[-Bin]);
    }
    
}

In [14]:
uncertainties::udouble GetKpipipiQCFactor(int Bin) {
    auto sqrtKK = uncertainties::sqrt(Ki[TMath::Abs(Bin)]*Kbari[TMath::Abs(Bin)]);
    auto cosDeltaD = uncertainties::cos(deltaD_Kpipipi);
    auto sinDeltaD = uncertainties::sin(deltaD_Kpipipi);
    if(Bin > 0) {
        auto Phases = ci[Bin]*cosDeltaD + si[Bin]*sinDeltaD;
        return 1 - 2.0*rD_Kpipipi*R_Kpipipi*sqrtKK*Phases/(Kbari[Bin] + rD2_Kpipipi*Ki[Bin]);
    } else {
        auto Phases = ci[-Bin]*cosDeltaD - si[-Bin]*sinDeltaD;
        return 1 - 2.0*rD_Kpipipi*R_Kpipipi*sqrtKK*Phases/(Ki[-Bin] + rD2_Kpipipi*Kbari[-Bin]);
    }
    
}

### Print results

In [21]:
std::vector<uncertainties::udouble> QCFactors;
for(int Bin = -4; Bin <= 4; Bin++) {
    if(Bin == 0) {
        continue;
    }
    std::string Label = "Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_Kpipipi_SignalBin";
    Label += std::string(Bin > 0 ? "P" : "M") + std::to_string(TMath::Abs(Bin)) + "_TagBin0";
    Label += "_QuantumCorrelationFactor";
    QCFactors.push_back(GetKSKpiQCFactor(Bin)/GetKpipipiQCFactor(Bin));
    std::cout << Label << " " << uncertainties::nom(QCFactors.back()) << "\n";
    std::cout << Label << "_err " << uncertainties::sdev(QCFactors.back()) << "\n";
}
std::string Filename = "PeakingBackground_DT_KSKpi_to_Kpipipi_QuantumCorrelationFactors.root";
std::vector<double> FlatCovMatrix = uncertainties::cov_matrix<std::vector<double>>(QCFactors);
SaveCovMatrix(FlatCovMatrix, Filename);

Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_Kpipipi_SignalBinM4_TagBin0_QuantumCorrelationFactor 1.26588
Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_Kpipipi_SignalBinM4_TagBin0_QuantumCorrelationFactor_err 0.125868
Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_Kpipipi_SignalBinM3_TagBin0_QuantumCorrelationFactor 0.373173
Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_Kpipipi_SignalBinM3_TagBin0_QuantumCorrelationFactor_err 0.125367
Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_Kpipipi_SignalBinM2_TagBin0_QuantumCorrelationFactor 0.439733
Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_Kpipipi_SignalBinM2_TagBin0_QuantumCorrelationFactor_err 0.0846187
Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_Kpipipi_SignalBinM1_TagBin0_QuantumCorrelationFactor 1.27528
Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_Kpipipi_SignalBinM1_TagBin0_QuantumCorrelationFactor_err 0.116898
Kpipipi_PeakingBackground1_DoubleTag_Flavour_KKpi